# Suite VLGRD — Reading Verilog with slang

The standards read source text through slang, a complete SystemVerilog front end: the conformance source reads to
a valid tree that prints and reads back the same; each construct reads to its kinds; directives, comments and macro
uses are kept where items and statements are listed; and what slang reports, the reader refuses or a standard lacks
fails with its line and column.

In [ ]:
import os
import tempfile

from mbse.Programs.Framework import Errors
from mbse.Programs.Framework.Syntax import same, walk
from mbse.Programs.Verilog import SystemVerilog2017, SystemVerilog2023, Syntax as S, Verilog2005, VerilogStandard
from mbse.Schemas.Framework import JSON
from support import raises

L = S.LANGUAGE
SOURCES = "../../conformance/sources"


def read(text, **options):
    return SystemVerilog2023.parse(text, **options)


def items(text):
    return read(f"module m;\n{text}\nendmodule\n").items[0].items


def statements(text):
    return read(f"module m;\ninitial begin\n{text}\nend\nendmodule\n").items[0].items[0].body.items


def expression(text):
    return statements(f"x = {text};")[0].value


def fails(text, message, error=Errors.ParseError, **options):
    with raises(error, match=message):
        read(text, **options)

## VLGRD-01 · The conformance source reads to a valid tree that prints and reads back the same

In [ ]:
text = open(f"{SOURCES}/systemverilog2023.sv", encoding="utf-8").read()
tree = read(text, include_paths=[SOURCES])
assert L.validate(tree) == [] and {type(n).__name__ for n in walk(tree)} == set(L.kinds())
printed = SystemVerilog2023.print(tree)
again = read(printed, include_paths=[SOURCES])
assert same(again, tree) and SystemVerilog2023.print(again) == printed
print(len(list(walk(tree))), "syntax nodes")

## VLGRD-02 · Design units: headers, imports, parameters, ANSI, interface and non-ANSI ports

In [ ]:
unit = read("""package p; endpackage
module automatic m import p::*; #(parameter int W = 8, localparam N = 2, parameter type T = logic) (
    input wire [W-1:0] a, output var logic b = 1'b0, inout tri c, ref int r, bus.mp port, interface generic [2]);
endmodule : m
macromodule n (a, b); input a; output reg [3:0] b; endmodule
interface automatic i; endinterface
program static q; endprogram
""")
m = unit.items[1]
assert (m.keyword, m.lifetime, m.name.spelling, m.labeled) == ("module", "automatic", "m", True)
assert m.imports[0].items[0].package.spelling == "p" and m.imports[0].items[0].name is None
parameter, local, kind = m.parameters
assert (parameter.keyword, parameter.type.keyword, parameter.assignments[0].name.spelling) == ("parameter", "int", "W")
assert local.keyword == "localparam" and local.type is None and local.assignments[0].value.spelling == "2"
assert isinstance(kind, S.TypeParameterDeclaration) and kind.assignments[0].type.keyword == "logic"
a, b, c, r, port, generic = m.ports
assert (a.direction, a.net_type, a.type.dimensions[0].left.left.name.spelling) == ("input", "wire", "W")
assert (b.direction, b.var, b.type.keyword, b.value.spelling) == ("output", True, "logic", "1'b0")
assert (c.net_type, r.direction, r.type.keyword) == ("tri", "ref", "int")
assert isinstance(port, S.InterfacePort) and (port.interface.spelling, port.modport.spelling) == ("bus", "mp")
assert generic.interface is None and generic.modport is None and generic.dimensions[0].size.spelling == "2"
n = unit.items[2]
assert n.keyword == "macromodule" and [p.name.spelling for p in n.ports] == ["a", "b"]
assert isinstance(n.items[0], S.PortDeclaration) and n.items[0].declarators[0].name.spelling == "a"
assert n.items[1].type.keyword == "reg" and n.items[1].direction == "output"
assert unit.items[3].lifetime == "automatic" and unit.items[4].lifetime == "static"
print("ok")

## VLGRD-03 · Data types and dimensions

In [ ]:
declared = items("""logic signed [7:0] a [4][0:1];
int unsigned b [string];
byte c [*];
shortreal d [$:8];
string e [];
p::t f;
struct packed signed { bit [3:0] x, y; } g;
union { int i; real r; } h;
enum { A, B = 2 } k;
event l;
realtime m;
chandle n;""")
a = declared[0]
assert (a.type.keyword, a.type.signing, a.type.dimensions[0].right.spelling) == ("logic", "signed", "0")
assert a.declarators[0].dimensions[0].size.spelling == "4" and isinstance(a.declarators[0].dimensions[1], S.RangeDimension)
assert declared[1].type.signing == "unsigned" and declared[1].declarators[0].dimensions[0].type.keyword == "string"
assert declared[2].declarators[0].dimensions[0].type is None
assert declared[3].type.keyword == "shortreal" and declared[3].declarators[0].dimensions[0].bound.spelling == "8"
assert isinstance(declared[4].declarators[0].dimensions[0], S.UnsizedDimension) and declared[4].type.keyword == "string"
assert (declared[5].type.name.scope.spelling, declared[5].type.name.name.spelling) == ("p", "t")
g = declared[6].type
assert (g.keyword, g.packed, g.signing, [d.name.spelling for d in g.members[0].declarators]) == ("struct", True, "signed", ["x", "y"])
assert declared[7].type.keyword == "union" and not declared[7].type.packed
k = declared[8].type
assert k.base is None and [m.name.spelling for m in k.members] == ["A", "B"] and k.members[1].value.spelling == "2"
assert [d.type.keyword for d in declared[9:]] == ["event", "realtime", "chandle"]
print("ok")

## VLGRD-04 · Declarations and items: nets, variables, typedefs, genvars, imports, modports, assigns and constructs

In [ ]:
declared = items("""wire [1:0] #3 w = 2'b01, v;
const var static int x = 1;
typedef logic [3:0] nibble_t [2];
genvar g, h;
import p::a, p::*;
assign #1 w = v, v = 0;
always @* x = 1;
always_comb x = 2;
always_latch x = 3;
initial x = 4;
final x = 5;
function automatic void f(); endfunction : f
function g2(input a); return a; endfunction
task t(output int o, input logic [1:0] i = 2'b0); endtask
task old; input a; endtask""")
wire, var, typedef, genvar, imports, assign = declared[:6]
assert (wire.net_type, wire.delay.value.spelling, wire.type.dimensions[0].left.spelling) == ("wire", "3", "1")
assert [d.name.spelling for d in wire.declarators] == ["w", "v"] and wire.declarators[0].value.spelling == "2'b01"
assert (var.const, var.var, var.lifetime) == (True, True, "static")
assert typedef.dimensions[0].size.spelling == "2" and [n.spelling for n in genvar.names] == ["g", "h"]
assert [(i.package.spelling, i.name and i.name.spelling) for i in imports.items] == [("p", "a"), ("p", None)]
assert assign.delay.value.spelling == "1" and len(assign.assignments) == 2
assert [c.keyword for c in declared[6:9]] == ["always", "always_comb", "always_latch"]
assert isinstance(declared[9], S.InitialConstruct) and isinstance(declared[10], S.FinalConstruct)
f, g2, t, old = declared[11:]
assert (f.lifetime, f.type.keyword, f.labeled, f.ports) == ("automatic", "void", True, [])
assert g2.type is None and g2.ports[0].direction == "input" and isinstance(g2.body[0], S.ReturnStatement)
assert t.ports[1].value.spelling == "2'b0" and t.ports[0].type.keyword == "int"
assert old.ports == [] and isinstance(old.body[0], S.PortDeclaration)
interface = read("interface i; modport mp (input a, output b, inout c); endinterface").items[0]
assert [(p.direction, p.name.spelling) for p in interface.items[0].items[0].ports] == [("input", "a"), ("output", "b"),
                                                                                      ("inout", "c")]
print("ok")

## VLGRD-05 · Generate constructs and instantiation

In [ ]:
declared = items("""generate for (genvar i = 0; i < 2; i++) begin : g end endgenerate
for (j = 0; j < 2; j = j + 1) assign a[j] = 0;
if (W) begin end else if (V) assign a = 1; else begin : e end : e
case (W) 1, 2: assign a = 1; default: begin end endcase
sub #(1, .N(2), .T(logic)) u1 (a, .b(c), .d(), .e), u2[1:0] (.*);""")
region = declared[0]
loop = region.items[0]
assert loop.genvar and loop.name.spelling == "i" and isinstance(loop.step, S.IncrementExpression) and loop.body.name.spelling == "g"
assert not declared[1].genvar and isinstance(declared[1].body, S.ContinuousAssign)
conditional = declared[2]
assert isinstance(conditional.alternative, S.GenerateIf) and conditional.alternative.alternative.labeled
case = declared[3]
assert [len(i.expressions) for i in case.items] == [2, 0]
instantiation = declared[4]
assert [type(p).__name__ for p in instantiation.parameters] == ["IntegerLiteral", "NamedConnection", "NamedConnection"]
assert instantiation.parameters[2].value.keyword == "logic"
u1, u2 = instantiation.instances
assert [type(c).__name__ for c in u1.connections] == ["NameExpression", "NamedConnection", "NamedConnection", "NamedConnection"]
assert u1.connections[2].value is None and not u1.connections[2].implicit and u1.connections[3].implicit
assert isinstance(u2.connections[0], S.WildcardConnection) and u2.dimensions[0].left.spelling == "1"
print("ok")

## VLGRD-06 · Statements

In [ ]:
body = statements("""a = 1; b <= #2 c; d += 1; e <= @(posedge clk) f;
f(); i++; --j; ;
begin : named int k; end : named
fork join_any
fork : par join_none
unique if (a) b = 1; else if (c) b = 2; else b = 3;
priority casez (a) inside 1, [2:3]: ; default b = 0; endcase
casex (a) 1: ; endcase
for (int k = 0, m = 1; k < 2; k++, m--) ;
for (;;) break;
while (a) continue;
repeat (2) ;
forever ;
do a = 1; while (b);
foreach (q[i, j]) a = 1;
#1 ;
@(a or posedge b, negedge c iff d) ;
@e a = 1;
@(*) ;
wait (a) ;
-> ev;
->> ev;
disable named;
disable fork;
assert (a) b = 1; else b = 2;
assume #0 (a);
cover final (a) b = 1;
return;""")
assignment = body[0]
assert (assignment.operator, assignment.timing) == ("=", None)
assert body[1].operator == "<=" and body[1].timing.value.spelling == "2"
assert body[2].operator == "+=" and body[3].timing.events[0].edge == "posedge"
assert isinstance(body[4].expression, S.CallExpression) and body[5].expression.postfix and not body[6].expression.postfix
assert isinstance(body[7], S.NullStatement)
named = body[8]
assert named.name.spelling == "named" and named.labeled and isinstance(named.items[0], S.VariableDeclaration)
assert body[9].join == "join_any" and body[10].name.spelling == "par" and body[10].join == "join_none"
conditional = body[11]
assert conditional.qualifier == "unique" and conditional.alternative.qualifier is None
assert isinstance(conditional.alternative.alternative, S.AssignmentStatement)
case = body[12]
assert (case.qualifier, case.keyword, case.inside) == ("priority", "casez", True)
assert isinstance(case.items[0].expressions[1], S.ValueRange) and case.items[1].expressions == []
assert body[13].keyword == "casex"
loop = body[14]
assert [type(i).__name__ for i in loop.initializers] == ["VariableDeclaration", "VariableDeclaration"] and len(loop.steps) == 2
assert loop.initializers[0].type.keyword == "int" and loop.initializers[1].type is None  # `m` is an int too
assert body[15].condition is None and body[15].initializers == []
assert [type(s).__name__ for s in body[16:21]] == ["WhileStatement", "RepeatStatement", "ForeverStatement",
                                                   "DoWhileStatement", "ForeachStatement"]
assert [v.spelling for v in body[20].variables] == ["i", "j"]
assert body[21].body is None and body[21].timing.value.spelling == "1"
events = body[22].timing.events
assert [(e.edge, e.expression.name.spelling) for e in events] == [(None, "a"), ("posedge", "b"), ("negedge", "c")]
assert events[2].condition.name.spelling == "d"
assert body[23].timing.events[0].expression.name.spelling == "e" and body[24].timing.events == []
assert body[25].body is None and not body[26].nonblocking and body[27].nonblocking
assert body[28].target.name.spelling == "named" and body[29].target is None
asserted = body[30]
assert asserted.pass_action.operator == "=" and asserted.fail_action.value.spelling == "2"
assert body[31].deferral == "#0" and body[31].keyword == "assume" and body[32].deferral == "final"
assert body[33].value is None
print("ok")

## VLGRD-07 · Expressions

In [ ]:
values = [expression(e) for e in [
    "a + b * c", "(a + b) * c", "-a", "&a", "~^a", "a ? b : c", "a inside {1, [2:3]}", "{a, b}", "{2{a}}",
    "'{1, 2}", "'{default: 0, a: 1, int: 2}", "t'{1}", "f(1, .b(2))", "$clog2(1)", "$time", "q.size()",
    "int'(x)", "8'(x)", "signed'(x)", "p::c", "s.m[1][7:0]", "a[i +: 2]", "a[i -: 2]", "q[$]", "8'hFF", "1.5e3",
    "10ns", "'1", "\"s\\n\"", "x++", "(y = 1)", "a <-> b"]]
assert values[0].operator == "+" and values[0].right.operator == "*"
assert isinstance(values[1].left, S.ParenthesizedExpression)
assert [v.operator for v in values[2:5]] == ["-", "&", "~^"]
assert isinstance(values[5], S.ConditionalExpression) and isinstance(values[6].set[1], S.ValueRange)
assert len(values[7].items) == 2 and values[8].count.spelling == "2"
assert values[9].items[1].spelling == "2" and values[9].type is None
assert [(i.key is None, type(i.key).__name__) for i in values[10].items] == [(True, "NoneType"), (False, "NameExpression"),
                                                                           (False, "IntegerAtomType")]
assert values[11].type.name.spelling == "t"
assert isinstance(values[12].arguments[1], S.NamedConnection) and values[13].name == "$clog2"
assert values[14].arguments == [] and values[15].callee.member.spelling == "size"
assert values[16].type.keyword == "int" and values[17].type.spelling == "8" and values[18].type.signing == "signed"
assert values[19].name.scope.spelling == "p"
member = values[20]
assert isinstance(member, S.RangeSelect) and isinstance(member.value, S.IndexExpression)
assert member.value.value.member.spelling == "m"
assert values[21].operator == "+:" and values[22].operator == "-:" and isinstance(values[23].index, S.DollarExpression)
assert [type(v).__name__ for v in values[24:29]] == ["IntegerLiteral", "RealLiteral", "TimeLiteral", "UnbasedUnsizedLiteral",
                                                     "StringLiteral"]
assert values[24].spelling == "8'hFF" and values[27].value == "1" and values[28].text == "s\\n"
assert values[29].postfix and isinstance(values[30].expression, S.AssignmentExpression) and values[31].operator == "<->"
print("ok")

## VLGRD-08 · Directives and comments where items and statements are listed; conditional compilation as a tree

In [ ]:
unit = read("""// one
`resetall
`define A 1
`define F(x, y) ((x) + (y))
`undef A
`default_nettype wire
`timescale 10ns / 1ns
`ifndef B
module m; /* inner */
`ifdef C
wire c;
`elsif D
wire d;
`else
wire e; // trailing
`endif
initial begin
`ifdef C
x = 1;
`endif
end
endmodule
`endif
""")
comment, resetall, define, function, undef, nettype, timescale, ifndef = unit.items
assert comment.text == " one" and not comment.trailing and resetall.text == "`resetall"
assert (define.name.spelling, define.body, define.function_like) == ("A", "1", False)
assert function.function_like and [p.spelling for p in function.parameters] == ["x", "y"]
assert undef.name.spelling == "A" and nettype.net_type == "wire" and (timescale.unit, timescale.precision) == ("10ns", "1ns")
assert ifndef.negated and ifndef.name.spelling == "B" and not ifndef.has_else
module = ifndef.items[0]
inner, ifdef = module.items[:2]
assert inner.block and inner.trailing
assert isinstance(ifdef.items[0], S.DisabledText) and ifdef.items[0].text == "wire c;"
assert isinstance(ifdef.branches[0].items[0], S.DisabledText) and ifdef.has_else
assert isinstance(ifdef.alternative[0], S.NetDeclaration) and ifdef.alternative[1].trailing
statement_ifdef = module.items[2].body.items[0]
assert isinstance(statement_ifdef, S.IfdefDirective) and statement_ifdef.items[0].text == "x = 1;"
taken = read("`ifdef C\nmodule m; endmodule\n`endif\n", defines=["C"])
assert isinstance(taken.items[0].items[0], S.ModuleDeclaration)
assert SystemVerilog2023.print(unit).startswith("// one\n`resetall\n`define A 1\n`define F(x, y) ((x) + (y))")
print("ok")

## VLGRD-09 · Macro uses and included files

In [ ]:
unit = read("""`define W 8
`define SUM(a, b) a + b
module m; wire [`W-1:0] x = `SUM(1, 2); endmodule
""")
declaration = unit.items[2].items[0]
width = declaration.type.dimensions[0].left.left
assert isinstance(width, S.MacroUsage) and width.name.spelling == "W" and width.arguments is None
value = declaration.declarators[0].value
assert isinstance(value, S.MacroUsage) and value.arguments == "1, 2"
with tempfile.TemporaryDirectory() as folder:
    with open(os.path.join(folder, "inc.svh"), "w") as f:
        f.write("`define FROM_INCLUDE 1\nwire included;\n")
    included = read('module m;\n`include "inc.svh"\nwire own;\nendmodule\n', include_paths=[folder])
    assert [type(i).__name__ for i in included.items[0].items] == ["IncludeDirective", "NetDeclaration"]
    assert included.items[0].items[1].declarators[0].name.spelling == "own"
fails('`include "missing.svh"\nmodule m; endmodule\n', "could not open include file")
print("ok")

## VLGRD-10 · What slang reports, what the reader does not support and what a standard lacks fail at their place

In [ ]:
fails("module m;\n  wire = ;\nendmodule\n", "line 2")
fails("module m; endmodule\nclass c; endclass\n", "unsupported syntax: ClassDeclaration")
fails("`define S a = 1;\nmodule m; initial begin `S end endmodule\n", "a macro use that is not a whole expression")
fails("module m; initial label: x = 1; endmodule\n", "a statement label")
fails("module m; wire [3:0] a = 1 + `undefined_macro; endmodule\n", "unknown directive")
with raises(Errors.ParseError, match="line 1, column 1: PackageDeclaration is not Verilog, but this is Verilog-2005"):
    Verilog2005.parse("package p; endpackage\n")
assert isinstance(Verilog2005.parse("module m (input a); endmodule\n").items[0], S.ModuleDeclaration)
error = None
try:
    read("module m;\n\n   wire = ;\nendmodule\n")
except Errors.ParseError as caught:
    error = caught
assert error is not None and (error.line, error.column) == (3, 9), (error.line, error.column)
print("ok")

## VLGRD-11 · What is outside the kinds is refused, naming it, where it is

In [ ]:
for source, message in [
    ("`define N n\nmodule m; wire `N; endmodule", "line 2, column 16: unsupported syntax: a macro use that is not a whole expression"),
    ("`ifdef (A && B)\nmodule m; endmodule\n`endif", "unsupported syntax: IfDefDirective"),
    ("`ifdef A\nmodule a;\n`else\nmodule b;\n`endif\nendmodule", "line 5, column 1: unexpected `endif"),
    ("module m (.*); endmodule", "unsupported syntax: WildcardPortList"),
    ("module m (a[0]); input [1:0] a; endmodule", "unsupported syntax: ImplicitNonAnsiPort"),
    ("module m (.p(a)); input a; endmodule", "unsupported syntax: ExplicitNonAnsiPort"),
    ("module m #(parameter type enum T = e) (); endmodule", "unsupported syntax: TypeParameterDeclaration"),
    ("module m; virtual bus v; endmodule", "unsupported syntax: VirtualInterfaceType"),
    ("module m; typedef struct { rand int a; } t; endmodule", "unsupported syntax: StructUnionMember"),
    ("module m; enum { A[2] } e; endmodule", "unsupported syntax: Declarator"),
    ("module m; wire [3+:2] a; endmodule", "unsupported syntax: AscendingRangeSelect"),
    ("module m; wire (strong0, weak1) a; endmodule", "unsupported syntax: NetDeclaration"),
    ("module m; wire vectored [1:0] a; endmodule", "unsupported syntax: NetDeclaration"),
    ("interface i; modport mp (import f); endinterface", "unsupported syntax: ModportSubroutinePortList"),
    ("interface i; modport mp (input .a(b)); endinterface", "unsupported syntax: ModportExplicitPort"),
    ("module m; assign (strong0, weak1) a = 1; endmodule", "unsupported syntax: ContinuousAssign"),
    ("module m; function int f(const ref int a); endfunction endmodule", "unsupported syntax: FunctionPort"),
    ("module m; u i(a, , b); endmodule", "unsupported syntax: EmptyPortConnection"),
    ("module m; initial if (a matches 1) ; endmodule", "unsupported syntax: ConditionalPredicate"),
    ("module m; initial case (a) matches 1: ; endcase endmodule", "unsupported syntax: CaseStatement"),
    ("module m; initial randcase 1: ; endcase endmodule", "unsupported syntax: RandCaseStatement"),
    ("module m; initial wait fork; endmodule", "unsupported syntax: WaitForkStatement"),
    ("module m; initial ->> #1 e; endmodule", "unsupported syntax: NonblockingEventTriggerStatement"),
    ("module m; initial assert property (a); endmodule", "unsupported syntax: AssertPropertyStatement"),
    ("module m; initial #(1:2:3) ; endmodule", "unsupported syntax: MinTypMaxExpression"),
    ("module m; initial ##1 ; endmodule", "unsupported syntax: CycleDelay"),
    ("module m; initial x = f(a, ,b); endmodule", "unsupported syntax: EmptyArgument"),
    ("module m; initial x = '{2{a}}; endmodule", "unsupported syntax: ReplicatedAssignmentPattern"),
    ("module m; initial x = null; endmodule", "unsupported syntax: NullLiteralExpression"),
    ("module m; initial x = {>>{a}}; endmodule", "unsupported syntax: StreamingConcatenationExpression"),
    ("module m; initial x = $root.a; endmodule", "unsupported syntax: RootScope"),
    ("module m; initial x = this; endmodule", "unsupported syntax: ThisHandle"),
    ("module m; initial x = new; endmodule", "unsupported syntax: NewClassExpression"),
    ("module m; initial x = p::q::a; endmodule", "unsupported syntax: ScopedName"),
    ("module m; initial x = q.sum with (item); endmodule", "unsupported syntax: ArrayOrRandomizeMethodExpression"),
    ("module m; typedef union tagged { int a; } t; endmodule", "unsupported syntax: UnionType"),
    ("module m; int [1:0] a; endmodule", "unsupported syntax: IntType"),
    ("module m; assign #(1,2) a = b; endmodule", "unsupported syntax: Delay3"),
    ("module m; initial #1step ; endmodule", "unsupported syntax: OneStepDelay"),
    ("`pragma protect begin_protected\n`pragma protect data_block\nabc\n`pragma protect end_protected\nmodule m; endmodule",
     "unsupported syntax: skipped text"),
    ("module a;\n`ifdef A\n`else\nendmodule\n`endif", "line 3, column 1: expected `endif"),
    ("module m (a); bus_if.mp a; endmodule", "unsupported syntax: InterfacePortHeader"),
    ("module m; typedef a.b t; endmodule", "unsupported syntax: ScopedName"),
    ("module m; initial foreach (q[, j]) x = 1; endmodule", "unsupported syntax: EmptyIdentifierName"),
    ("module m; initial x = f(a[*2]); endmodule", "unsupported syntax: SimpleSequenceExpr"),
    ("module m; initial x = a.b#(1); endmodule", "unsupported syntax: ScopedName"),
    ("module m; initial x = a inside {[1 +/- 2]}; endmodule", "unsupported syntax: ValueRangeExpression"),
]:
    fails(source + "\n", message)
print("ok")

## VLGRD-12 · Corner cases the kinds hold

In [ ]:
unit = read('''// é: positions count characters, not bytes
module m #(parameter type T, parameter W) ();
logic a [int]; logic b [t]; automatic int c;
function static int f(); endfunction
if (1) begin end
u #(.N()) i(); u j();
initial begin
    for (i = 0, j = 1; i < 2; i++) ;
    @(posedge a or b iff c) ;
    @(e.triggered) ;
    x = a.b[1].c + a.b.c + t::a;
    x = '{a, b} + q.sum() + int'(1) + signed'(1) + a.b();
    x = $bits(int) + $bits(logic [1:0]) + f()[0] + f(a).b + f(.a(), .b(c));
    x: begin end
    for (int k; k < 2; k++) ;
    assert (a);
    x = """multi
line""";
end
if (1) g: begin end
endmodule
''')
module = unit.items[1]
assert unit.items[0].text == " é: positions count characters, not bytes"
assert module.parameters[0].assignments[0].type is None and module.parameters[1].assignments[0].value is None
assert module.items[0].declarators[0].dimensions[0].type.keyword == "int"
assert module.items[1].declarators[0].dimensions[0].size.name.spelling == "t"
assert module.items[2].lifetime == "automatic" and module.items[3].lifetime == "static"
assert module.items[5].parameters[0].value is None and module.items[6].instances[0].connections == []
body = module.items[7].body.items
assert len(body[0].initializers) == 2 and body[1].timing.events[1].condition.name.spelling == "c"
assert [type(a).__name__ for a in body[5].value.left.left.left.left.arguments] == ["IntegerAtomType"]
assert body[6].name.spelling == "x" and body[7].initializers[0].declarators[0].value is None
assert module.items[8].consequence.name.spelling == "g"
assert body[8].pass_action is None and body[8].fail_action is None
string = body[9].value
assert string.triple and string.text == "multi\nline"
with raises(Errors.ParseError, match="StringLiteral.triple needs SystemVerilog-2023, but this is SystemVerilog-2017"):
    SystemVerilog2017.parse('module m; initial x = """s"""; endmodule\n')
assert same(read(SystemVerilog2023.print(unit)), unit)
fails("// é\nmodule m; wire = ; endmodule\n", "line 2, column 16")
print("ok")


## VLGRD-13 · Reading for the other implementation: a request in, a tree or an error out, as JSON

In [ ]:
import json
import subprocess
import sys

from mbse.Programs.Verilog import read as reader

request = {"text": "module m; wire a; endmodule\n", "year": 2023, "family": "SystemVerilog", "include_paths": [], "defines": []}
response = json.loads(reader.respond(json.dumps(request)))
assert list(response) == ["tree"]
tree = JSON.FromJSON(S.LANGUAGE.Builders).Reachable(S.SourceText.Schema, response["tree"])
assert same(tree, read(request["text"]))
failed = json.loads(reader.respond(json.dumps({**request, "text": "module m;\n wire = ;\nendmodule\n"})))
assert failed == {"error": "expected declarator", "line": 2, "column": 7}, failed
lacking = json.loads(reader.respond(json.dumps({**request, "year": 2005, "family": "Verilog", "text": "package p; endpackage\n"})))
assert lacking["error"] == "PackageDeclaration is not Verilog, but this is Verilog-2005"
run = subprocess.run([sys.executable, "-m", "mbse.Programs.Verilog.read"], input=json.dumps(request).encode("utf-8"),
                     capture_output=True, check=True)
assert json.loads(run.stdout.decode("utf-8")) == response
print("ok")
